<a href="https://colab.research.google.com/github/AnnaBastin/Company-WikiBot/blob/main/company.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q -U langchain langchain-community langchain-text-splitters chromadb
!pip install -q google-generativeai langchain-google-genai
!pip install -q pypdf python-docx docx2txt
!pip install -U langchain-google-genai google-generativeai
!pip install python-dotenv

In [9]:
import os
import glob

from google.colab import files

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

from langchain_community.vectorstores import Chroma

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

import google.generativeai as genai

In [10]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
uploaded = files.upload()

os.makedirs("documents", exist_ok=True)

for filename in uploaded.keys():
    os.rename(filename, f"documents/{filename}")

print(os.listdir("documents"))

Saving employeebenefits.txt to employeebenefits.txt
Saving fakedoc.txt to fakedoc.txt
Saving leavepolicy.txt to leavepolicy.txt
['leavepolicy.txt', 'fakedoc.txt', 'employeebenefits.txt']


In [11]:
DOCUMENTS_PATH = "documents"
documents = []

all_files = glob.glob(f"{DOCUMENTS_PATH}/**/*", recursive=True)

print("Files found:")
for f in all_files:
    print(f)

for file_path in all_files:

    if os.path.isdir(file_path):
        continue

    try:
        if file_path.endswith(".txt"):
            loader = TextLoader(file_path)

        elif file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)

        elif file_path.endswith(".docx"):
            loader = Docx2txtLoader(file_path)

        else:
            print("Skipping:", file_path)
            continue

        documents.extend(loader.load())
        print("Loaded:", file_path)

    except Exception as e:
        print("Error:", file_path, e)

print("Total docs:", len(documents))

Files found:
documents/leavepolicy.txt
documents/fakedoc.txt
Loaded: documents/leavepolicy.txt
Loaded: documents/fakedoc.txt
Total docs: 2


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 2


In [13]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

models = genai.list_models()

for m in models:
    print("MODEL:", m.name)
    print("SUPPORTED METHODS:", m.supported_generation_methods)
    print("-" * 50)

for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print("EMBEDDING MODEL:", m.name)

MODEL: models/gemini-2.5-flash
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.5-pro
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-001
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-lite-001
SUPPORTED METHODS: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------
MODEL: models/gemini-2.0-flash-lite
SUPPORTED METHODS: ['generateContent',

Forbidden: 403 GET https://generativelanguage.googleapis.com/v1beta/models?pageSize=50&%24alt=json%3Benum-encoding%3Dint: Your API key was reported as leaked. Please use another API key.

In [14]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

test = embeddings.embed_query("hello world")
print("Embedding length:", len(test))

GoogleGenerativeAIError: Error embedding content (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}

In [15]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)

retriever = vectorstore.as_retriever()

GoogleGenerativeAIError: Error embedding content (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}

In [16]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [17]:

if not retrieved_docs:
    print("No relevant documents found.")
else:
    context = "\n".join([doc.page_content for doc in retrieved_docs])

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

#output = response.content
clean_text = response.content[0]["text"]
print(clean_text)
#print(output)

NameError: name 'retrieved_docs' is not defined